In [28]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("vbb-data-validation")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)

#spark.range(10).show()

Spark version: 3.5.8
Spark master: spark://spark-master:7077


## Load and inspect data

In [18]:
from pathlib import Path
from pyspark.sql import functions as F

DATA_PATH = Path(
    "/opt/spark/work-dir/data/historical_delays/data-2024-07.parquet"
)

print("File exists:", DATA_PATH.exists())
print("File size MB:", round(DATA_PATH.stat().st_size / 1024**2, 2))

File exists: True
File size MB: 101.87


In [19]:
spark.conf.set(
    "spark.sql.legacy.parquet.nanosAsLong",
    "true",
)

print(
    spark.conf.get(
        "spark.sql.legacy.parquet.nanosAsLong"
    )
)

true


In [20]:
historical_df = spark.read.parquet(str(DATA_PATH))
historical_df.printSchema()

root
 |-- station_name: string (nullable = true)
 |-- xml_station_name: string (nullable = true)
 |-- eva: string (nullable = true)
 |-- train_number: string (nullable = true)
 |-- line_number: string (nullable = true)
 |-- final_destination_station: string (nullable = true)
 |-- delay_in_min: integer (nullable = true)
 |-- time: long (nullable = true)
 |-- is_canceled: boolean (nullable = true)
 |-- train_type: string (nullable = true)
 |-- train_line_ride_id: string (nullable = true)
 |-- train_line_station_num: integer (nullable = true)
 |-- arrival_planned_time: long (nullable = true)
 |-- arrival_change_time: long (nullable = true)
 |-- departure_planned_time: long (nullable = true)
 |-- departure_change_time: long (nullable = true)
 |-- id: string (nullable = true)



In [21]:
historical_df.show(
    2,
    truncate=False,
    vertical=True,
)

-RECORD 0-------------------------------------------------------
 station_name              | NULL                               
 xml_station_name          | ZOB/Hauptbahnhof, Pforzheim        
 eva                       | 0940370                            
 train_number              | 33382                              
 line_number               | S6 (S                              
 final_destination_station | Bahnhof, Bad Wildbad               
 delay_in_min              | 0                                  
 time                      | 1719792000000000000                
 is_canceled               | false                              
 train_type                | Bus                                
 train_line_ride_id        | -6129702905591104469               
 train_line_station_num    | 1                                  
 arrival_planned_time      | NULL                               
 arrival_change_time       | NULL                               
 departure_planned_time  

In [22]:
print("Columns:", len(historical_df.columns))
row_count = historical_df.count()
print(f"Rows: {row_count:,}")

# A partition is a chunk of the DataFrame that Spark can process independently.
# Each partition usually becomes one Spark task. Spark can process several partitions in parallel.
print("Partitions:", historical_df.rdd.getNumPartitions()) 

Columns: 17
Rows: 2,007,251
Partitions: 10


## Convert timestamp
A timestamp such as: 2024-07-01 12:30:00 is stored in the Parquet file as nanoseconds: 1719837000000000000

Spark provides timestamp_micros() for converting epoch microseconds to a Spark timestamp. Because the source values are nanoseconds, they must first be divided by 1,000.


In [25]:
# Use UTC internally for consistent pipeline processing.
spark.conf.set("spark.sql.session.timeZone", "UTC")

timestamp_columns = [
    "time",
    "arrival_planned_time",
    "arrival_change_time",
    "departure_planned_time",
    "departure_change_time",
]

historical_df_converted = historical_df

for column_name in timestamp_columns:
    historical_df_converted = historical_df_converted.withColumn(
        f"{column_name}_converted",
        F.when(
            F.col(column_name).isNotNull(),
            F.timestamp_micros(
                (F.col(column_name) / F.lit(1_000)).cast("long")
            ),
        ).otherwise(F.lit(None).cast("timestamp")),
    )

In [27]:
# Inspect the converted values:
historical_df_converted.select(
    "station_name",
    "xml_station_name",
    "time",
    "time_converted",
    "arrival_planned_time",
    "arrival_planned_time_converted",
    "arrival_change_time",
    "arrival_change_time_converted",
    "departure_planned_time",
    "departure_planned_time_converted",
    "departure_change_time",
    "departure_change_time_converted",
).show(
    1,
    truncate=False,
    vertical=True,
)

-RECORD 0-------------------------------------------------------
 station_name                     | NULL                        
 xml_station_name                 | ZOB/Hauptbahnhof, Pforzheim 
 time                             | 1719792000000000000         
 time_converted                   | 2024-07-01 00:00:00         
 arrival_planned_time             | NULL                        
 arrival_planned_time_converted   | NULL                        
 arrival_change_time              | NULL                        
 arrival_change_time_converted    | NULL                        
 departure_planned_time           | 1719792000000000000         
 departure_planned_time_converted | 2024-07-01 00:00:00         
 departure_change_time            | 1719792000000000000         
 departure_change_time_converted  | 2024-07-01 00:00:00         
only showing top 1 row

